In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ===============================
# 1) Veri yükleme
# ===============================
df = pd.read_csv("BostonHousing.csv")

# Hedef sütunu adı: genelde 'MEDV'
target_col = "MEDV" if "MEDV" in df.columns else df.columns[-1]

X = df.drop(columns=[target_col])
y = df[target_col]

# ===============================
# 2) Train/Test ayırma
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===============================
# 3) Değerlendirme yardımcıları
# ===============================
def evaluate(model, X_tr, y_tr, X_te, y_te, name="model"):
    y_pred = model.predict(X_te)
    r2  = r2_score(y_te, y_pred)
    rmse = mean_squared_error(y_te, y_pred, squared=False)
    mae = mean_absolute_error(y_te, y_pred)
    print(f"\n{name} -> Test R^2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
    return {"name": name, "r2": r2, "rmse": rmse, "mae": mae}

def coef_table(fitted_pipeline, feature_names):
    # Pipeline => ... -> scaler -> model
    model = fitted_pipeline.named_steps["model"]
    coefs = getattr(model, "coef_", None)
    if coefs is None:
        return None
    tbl = pd.DataFrame({"feature": feature_names, "coef": coefs})
    tbl["abs_coef"] = tbl["coef"].abs()
    return tbl.sort_values("abs_coef", ascending=False).drop(columns=["abs_coef"])

# ===============================
# 4) Ortak Pipeline iskeleti
# - Median impute (eksikler)
# - Standartlaştırma (Ridge/Lasso/EN için önemli)
# - Model
# ===============================
base_steps = [
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),  # yer tutucu, GridSearch'te değişecek
]

# ===============================
# 5) Modeller ve arama ızgarası
# ===============================
models_and_grids = [
    (
        "LinearRegression (baseline)",
        Pipeline(steps=base_steps[:-1] + [("model", LinearRegression())]),
        {}  # hiperparametre yok
    ),
    (
        "Ridge",
        Pipeline(steps=base_steps[:-1] + [("model", Ridge())]),
        {
            "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
        }
    ),
    (
        "Lasso",
        Pipeline(steps=base_steps[:-1] + [("model", Lasso(max_iter=20000))]),
        {
            "model__alpha": [0.0005, 0.001, 0.01, 0.1, 1.0, 10.0],
            "model__max_iter": [20000]
        }
    ),
    (
        "ElasticNet",
        Pipeline(steps=base_steps[:-1] + [("model", ElasticNet(max_iter=20000))]),
        {
            "model__alpha": [0.0005, 0.001, 0.01, 0.1, 1.0, 10.0],
            "model__l1_ratio": [0.2, 0.5, 0.8],
            "model__max_iter": [20000]
        }
    ),
]

# CV ayarı
cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = []
best_model = None
best_rmse = float("inf")
best_name = None

# ===============================
# 6) Her model için GridSearchCV ve değerlendirme
# ===============================
for name, pipe, grid in models_and_grids:
    if grid:
        gs = GridSearchCV(
            estimator=pipe,
            param_grid=grid,
            cv=cv,
            scoring="neg_root_mean_squared_error",  # RMSE'yi maksimize etmek = -RMSE'yi maksimize etmek
            n_jobs=-1,
            refit=True
        )
        gs.fit(X_train, y_train)
        print(f"\n{name} -> En iyi parametreler: {gs.best_params_}")
        fitted = gs.best_estimator_
    else:
        # LinearRegression baseline
        pipe.fit(X_train, y_train)
        fitted = pipe
        print(f"\n{name} -> (parametre yok)")
    
    # Test set performansı
    metrics = evaluate(fitted, X_train, y_train, X_test, y_test, name=name)
    results.append(metrics)

    # En iyi RMSE'yi takip et
    if metrics["rmse"] < best_rmse:
        best_rmse = metrics["rmse"]
        best_model = fitted
        best_name = name

# ===============================
# 7) Sonuç özeti
# ===============================
res_df = pd.DataFrame(results).sort_values("rmse")
print("\n=== Model karşılaştırma (RMSE küçük daha iyi) ===")
print(res_df.to_string(index=False))

# ===============================
# 8) En iyi modelin katsayıları
# ===============================
print(f"\nEn iyi model: {best_name}")
coef_df = coef_table(best_model, X.columns)
if coef_df is not None:
    print("\nKatsayılar (mutlak değere göre sıralı):")
    print(coef_df.head(15).to_string(index=False))
else:
    print("Bu model lineer değil veya katsayı üretmiyor.")


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\523579\AppData\Roaming\Python\Python39\site-packages\IPython\core\interactiveshell.py", line 3550, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\523579\AppData\Local\Temp\ipykernel_129272\3709496615.py", line 4, in <module>
    from sklearn.model_selection import train_test_split, GridSearchCV, KFold
  File "C:\Users\523579\AppData\Roaming\Python\Python39\site-packages\sklearn\__init__.py", line 82, in <module>
    from .base import clone
  File "C:\Users\523579\AppData\Roaming\Python\Python39\site-packages\sklearn\base.py", line 17, in <module>
    from .utils import _IS_32BIT
  File "C:\Users\523579\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\__init__.py", line 29, in <module>
    from .fixes import parse_version, threadpool_info
  File "C:\Users\523579\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\fixes.py", line 19, in <module>
    import scipy.stats
  File "C:\Use